# 07 -- Publication-Ready PDF Reports

`venn-diagram-lab` can bundle every view into a single multi-page PDF:
the overview pie chart, the Venn + UpSet panel, pairwise statistics tables,
the network graph, and a methodology appendix.

**What you will learn:**

- Generate a full PDF report with one method call
- Inspect file size and verify the PDF magic bytes
- Understand the multi-page layout and when it expands
- Opt out of the network or about pages for a smaller file
- Use the standalone `vdl.render_pdf_report()` function form


In [ ]:
import venn_diagram_lab as vdl

print(f'venn-diagram-lab {vdl.__version__}')

## Generate the report

In [ ]:
result = vdl.analyze(
    vdl.load_sample('dataset_real_cancer_drivers_4'),
    model='auto',
)

result.to_pdf_report(
    '/tmp/cancer_drivers.pdf',
    title='Cancer Drivers Comparison',
)
print('Report written to /tmp/cancer_drivers.pdf')

In [ ]:
from pathlib import Path

p = Path('/tmp/cancer_drivers.pdf')
size_kb = p.stat().st_size / 1024
magic = p.read_bytes()[:4]
print(f'File size : {size_kb:.1f} KB')
print(f'Magic bytes: {magic}  (valid PDF starts with %PDF)')

## Page layout

The report contains between 4 and 7 pages depending on the number of sets:

| Page | Content | Notes |
|------|---------|-------|
| 1 | **Overview** -- title, pie chart of set sizes, per-set item count table, metadata (model, sets, universe size) | Always present |
| 2 | **Venn + UpSet** -- Venn SVG (left) and UpSet plot (right) side by side at 300 dpi | Always present |
| 3 | **Statistics** -- Jaccard (top-left), Dice (top-right), and Hypergeometric + BH-FDR (bottom) on one page for 2-6 sets | 1 page for 2-6 sets; 3 separate pages for 7-9 sets |
| 4 | **Network** -- force-directed set relationship graph + significant edge list | Skipped when `include_network=False` |
| 5 | **About** -- methodology appendix (Venn, UpSet, Jaccard, Dice, enrichment) | Skipped when `include_about=False` |

**Summary:**

- 2-6 sets, all pages: **5 pages** (1 overview + 1 venn/upset + 1 stats + 1 network + 1 about)
- 7-9 sets, all pages: **7 pages** (same, but statistics expand to 3 pages)
- Minimal (no network, no about): **3 pages** for 2-6 sets


## Skip network or about pages

In [ ]:
result.to_pdf_report(
    '/tmp/r_minimal.pdf',
    include_network=False,
    include_about=False,
)

full_kb = Path('/tmp/cancer_drivers.pdf').stat().st_size / 1024
mini_kb = Path('/tmp/r_minimal.pdf').stat().st_size / 1024
print(f'Full report  : {full_kb:.1f} KB')
print(f'Minimal report: {mini_kb:.1f} KB')
print(f'Size reduction: {(1 - mini_kb / full_kb) * 100:.0f}%')

## Standalone function form

In [ ]:
vdl.render_pdf_report(result, '/tmp/r2.pdf', title='Cancer Drivers Comparison')

size_kb = Path('/tmp/r2.pdf').stat().st_size / 1024
print(f'r2.pdf: {size_kb:.1f} KB')
print('render_pdf_report() is identical to result.to_pdf_report()')

## v2.2.3 additions: share distribution + cluster heatmap

Two PDF additions land in v2.2.3 to close the webtool parity gap:

1. **Item Share Distribution page** -- always present. Adds the
   histogram (items shared by exactly k sets, for k=1..N) plus a
   per-bin breakdown table. Sits between the Statistics tables and
   the Network page.
2. **Cluster-mode heatmap** -- opt-in via `cluster_heatmap=True`.
   Appends a cluster-ordered Jaccard similarity heatmap (UPGMA
   average linkage by default) with L-shaped dendrograms,
   mirroring the webtool's `axisOrder=cluster` toggle on the PDF.


In [ ]:
result.to_pdf_report('/tmp/r_cluster.pdf', cluster_heatmap=True)

default_kb = Path('/tmp/cancer_drivers.pdf').stat().st_size / 1024
cluster_kb = Path('/tmp/r_cluster.pdf').stat().st_size / 1024
print(f'Default report   : {default_kb:.1f} KB')
print(f'+ cluster heatmap : {cluster_kb:.1f} KB')
print(f'Extra page adds   : {cluster_kb - default_kb:.1f} KB')

## Next steps

- [`08_custom_styling_and_export.ipynb`](08_custom_styling_and_export.ipynb) -- SVG/PNG export and custom diagram styling for publication figures
